# Participant-Level Data Splitting Strategy

This notebook implements the participant-level data splitting strategy for the PADS movement metadata dataset. The objective is to prepare the dataset for machine learning by dividing participants into training, validation, and testing sets while preventing participant-level data leakage.

The splitting strategy ensures that all recordings belonging to the same participant remain within a single dataset, allowing unbiased model training and evaluation.

In [37]:
# ============================================================================
# Import Libraries
# ============================================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

# Set random seed
np.random.seed(42)

In [38]:
# ============================================================================
# Load Movement Metadata
# ============================================================================

DATA_PATH = Path("../data/processed")

movement_df = pd.read_csv(DATA_PATH / "movement_metadata.csv")

movement_df.head()

,patient_id,device,sampling_rate,task,samples,left_file,right_file
0,1,Apple Watch Series 4,100,CrossArms,1024,timeseries/001_CrossArms_LeftWrist.txt,timeseries/001_CrossArms_RightWrist.txt
1,1,Apple Watch Series 4,100,DrinkGlas,1024,timeseries/001_DrinkGlas_LeftWrist.txt,timeseries/001_DrinkGlas_RightWrist.txt
2,1,Apple Watch Series 4,100,Entrainment,2048,timeseries/001_Entrainment_LeftWrist.txt,timeseries/001_Entrainment_RightWrist.txt
3,1,Apple Watch Series 4,100,HoldWeight,1024,timeseries/001_HoldWeight_LeftWrist.txt,timeseries/001_HoldWeight_RightWrist.txt
4,1,Apple Watch Series 4,100,LiftHold,1024,timeseries/001_LiftHold_LeftWrist.txt,timeseries/001_LiftHold_RightWrist.txt


In [39]:
# =============================================================================
# Dataset Summary
# =============================================================================

print(f"Movement recordings : {len(movement_df):,}")
print(f"Participants        : {movement_df['patient_id'].nunique():,}")
print(f"Motor tasks         : {movement_df['task'].nunique():,}")

Movement recordings : 5,159
Participants        : 469
Motor tasks         : 11


In [40]:
# =============================================================================
# Dataset Structure
# =============================================================================

movement_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5159 entries, 0 to 5158
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   patient_id     5159 non-null   int64 
 1   device         5159 non-null   object
 2   sampling_rate  5159 non-null   int64 
 3   task           5159 non-null   object
 4   samples        5159 non-null   int64 
 5   left_file      5159 non-null   object
 6   right_file     5159 non-null   object
dtypes: int64(3), object(4)
memory usage: 282.3+ KB


In [41]:
# =============================================================================
# Check Unique Participants
# =============================================================================

movement_df["patient_id"].nunique()

469

In [42]:
# =============================================================================
# Split Participants into Training and Temporary Sets
# =============================================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    gss.split(
        movement_df,
        groups=movement_df["patient_id"]
    )
)

train_df = movement_df.iloc[train_idx].reset_index(drop=True)
temp_df = movement_df.iloc[temp_idx].reset_index(drop=True)

In [43]:
# =============================================================================
# Verify First Split
# =============================================================================

print(f"Training recordings : {len(train_df):,}")
print(f"Temporary recordings: {len(temp_df):,}")

print()

print(f"Training participants : {train_df['patient_id'].nunique()}")
print(f"Temporary participants: {temp_df['patient_id'].nunique()}")

Training recordings : 3,608
Temporary recordings: 1,551

Training participants : 328
Temporary participants: 141


In [44]:
# =============================================================================
# Split Temporary Set into Validation and Test Sets
# =============================================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss.split(
        temp_df,
        groups=temp_df["patient_id"]
    )
)

validation_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

In [45]:
# =============================================================================
# Verify Final Split
# =============================================================================

print(f"Training participants   : {train_df['patient_id'].nunique()}")
print(f"Validation participants : {validation_df['patient_id'].nunique()}")
print(f"Test participants       : {test_df['patient_id'].nunique()}")

print()

print(f"Training recordings   : {len(train_df):,}")
print(f"Validation recordings : {len(validation_df):,}")
print(f"Test recordings       : {len(test_df):,}")

Training participants   : 328
Validation participants : 70
Test participants       : 71

Training recordings   : 3,608
Validation recordings : 770
Test recordings       : 781


### Split Summary

The participant-level splitting strategy successfully divided the dataset into approximately 70% training, 15% validation, and 15% testing participants. The distribution of recordings across the three datasets is consistent with the participant allocation and provides sufficient data for model training, validation, and final performance evaluation.

In [46]:
# =============================================================================
# Check for Data Leakage
# =============================================================================

train_patients = set(train_df["patient_id"])
validation_patients = set(validation_df["patient_id"])
test_patients = set(test_df["patient_id"])

print("Train ∩ Validation:", len(train_patients & validation_patients))
print("Train ∩ Test      :", len(train_patients & test_patients))
print("Validation ∩ Test :", len(validation_patients & test_patients))

Train ∩ Validation: 0
Train ∩ Test      : 0
Validation ∩ Test : 0


### Data Leakage Summary
A participant-level data splitting strategy was successfully implemented using GroupShuffleSplit. The movement metadata dataset was divided into training, validation, and testing datasets while preserving participant independence. Verification confirmed that there was no overlap of participant IDs across the three datasets, eliminating participant-level data leakage. The resulting datasets are now prepared for feature engineering, model development, and performance evaluation.

In [47]:
# =============================================================================
# Save Split Datasets
# =============================================================================

OUTPUT_PATH = Path("../data/processed")

train_df.to_csv(OUTPUT_PATH / "train_metadata.csv", index=False)
validation_df.to_csv(OUTPUT_PATH / "validation_metadata.csv", index=False)
test_df.to_csv(OUTPUT_PATH / "test_metadata.csv", index=False)

print("Datasets saved successfully!")

Datasets saved successfully!


### Conclusion

A participant-level data splitting strategy was successfully implemented using GroupShuffleSplit. The movement metadata dataset was divided into training, validation, and testing datasets while preserving participant independence. The final verification confirmed that no participant appears in more than one dataset, making the data suitable for feature engineering and subsequent machine learning model development.